# 21 · Construction du rapport Power BI par l'API Fabric

Ce notebook relie le rapport « Demo mobilite Rouen » au modèle sémantique, lit sa définition (format PBIR), puis génère les visuels des quatre pages : « Qui a accès ? », « Qui est concerné ? », « Où investir ? » et « Le modèle est-il fiable ? ». Il utilise l'API REST de Fabric par la bibliothèque semantic-link (`sempy`).

Les identifiants du rapport (`rid`) et du modèle sémantique (`did`) sont propres à l'espace de travail de l'équipe : remplacez-les par les vôtres pour le relancer.

*Note de mise en forme :* dans l'export d'origine, le code des cellules était compressé (base64 et zlib). Il est ici décompressé, sans changement, pour être lisible. Les cellules d'exploration interrompues ont été retirées.

## 1. Relier le rapport au modèle sémantique

In [ ]:
import sempy.fabric as fabric
ws = fabric.get_workspace_id()
rid = "d849391a-78c1-434c-9cca-3ea264616612"
did = "194a4133-75a2-4137-a8c7-63cdec66b193"
r = fabric.PowerBIRestClient().post(f"v1.0/myorg/groups/{ws}/reports/{rid}/Rebind", json={"datasetId": did})
print(r.status_code, r.text)

200 


## 2. Lire la définition du rapport

In [ ]:
import base64, json
c = fabric.FabricRestClient()
res = c.post(f"v1/workspaces/{ws}/reports/{rid}/getDefinition", lro_wait=True)
defn = res.json()["definition"]
parts = {p["path"]: base64.b64decode(p["payload"]).decode("utf-8") for p in defn["parts"]}
print(defn.get("format"), [ (k, len(v)) for k, v in parts.items()])

PBIR [('definition.pbir', 411), ('StaticResources/SharedResources/BaseThemes/Fluent2-CY26SU10.json', 95296), ('StaticResources/RegisteredResources/Mobilite_Rouen5791791956429733.json', 8946), ('definition/version.json', 149), ('definition/report.json', 1695), ('definition/pages/pages.json', 302), ('definition/pages/df1af542e16e95ba8af9/page.json', 251), ('definition/pages/df1af542e16e95ba8af9/visuals/e059da448c80fd2c0178/visual.json', 4272), ('definition/pages/df1af542e16e95ba8af9/visuals/ca87ce54788831125d38/visual.json', 1299), ('definition/pages/df1af542e16e95ba8af9/visuals/9395e5f5d754c6e80349/visual.json', 664), ('definition/pages/df1af542e16e95ba8af9/visuals/144c2828c63f13db4b4f/visual.json', 1650), ('definition/pages/df1af542e16e95ba8af9/visuals/78fdba26a8a43d0953f4/visual.json', 2589), ('definition/pages/9094c6b5bd99cc1d233f/page.json', 256), ('definition/pages/9094c6b5bd99cc1d233f/visuals/7bf067c6902d4b70d893/visual.json', 3040), ('definition/pages/9094c6b5bd99cc1d233f/visuals

In [ ]:
for k, v in parts.items():
    if k.endswith("page.json") or k.endswith("pages.json"): print(k, v.replace(chr(10), " ")[:400])
    

definition/pages/pages.json {   "$schema": "https://developer.microsoft.com/json-schemas/fabric/item/report/definition/pagesMetadata/1.1.0/schema.json",   "pageOrder": [     "df1af542e16e95ba8af9",     "9094c6b5bd99cc1d233f",     "fa72a2877f335b1fa1b7",     "a99bf06cedda2f4d52a6"   ],   "activePageName": "a99bf06cedda2f4d52a6" }
definition/pages/df1af542e16e95ba8af9/page.json {   "$schema": "https://developer.microsoft.com/json-schemas/fabric/item/report/definition/page/2.1.0/schema.json",   "name": "df1af542e16e95ba8af9",   "displayName": "Qui a accès ?",   "displayOption": "FitToPage",   "height": 1080,   "width": 1920 }
definition/pages/9094c6b5bd99cc1d233f/page.json {   "$schema": "https://developer.microsoft.com/json-schemas/fabric/item/report/definition/page/2.1.0/schema.json",   "name": "9094c6b5bd99cc1d233f",   "displayName": "Qui est concerné ?",   "displayOption": "FitToPage",   "height": 1080,   "width": 1920 }
definition/pages/fa72a2877f335b1fa1b7/page.json {   "$schema": "

## 3. Générer les visuels des quatre pages

Cartes Azure Maps, segments (scénario, type, budget), cartes de chiffres, tableaux et graphiques, puis envoi de la nouvelle définition (`updateDefinition`).

In [ ]:
import json, base64, traceback, uuid
import sempy.fabric as fabric
SCHEMA = "https://developer.microsoft.com/json-schemas/fabric/item/report/definition/visualContainer/2.14.0/schema.json"
ws = fabric.get_workspace_id()
rid = "d849391a-78c1-434c-9cca-3ea264616612"
c = fabric.FabricRestClient()
res = c.post(f"v1/workspaces/{ws}/reports/{rid}/getDefinition", lro_wait=True)
parts = {p["path"]: base64.b64decode(p["payload"]).decode("utf-8") for p in res.json()["definition"]["parts"]}

P1, P2, P3, P4 = "df1af542e16e95ba8af9", "9094c6b5bd99cc1d233f", "fa72a2877f335b1fa1b7", "a99bf06cedda2f4d52a6"
OLD = ["e059da448c80fd2c0178", "7bf067c6902d4b70d893", "a21a1e72362152e9b558"]
GEN = "cc"  # prefix of generated visuals (re-run safe)
for k in list(parts):
    if any(o in k for o in OLD) or ("/visuals/" + GEN) in k:
        del parts[k]

def lit(v):
    if isinstance(v, bool): return {"expr": {"Literal": {"Value": "true" if v else "false"}}}
    if isinstance(v, (int, float)): return {"expr": {"Literal": {"Value": f"{v}D"}}}
    return {"expr": {"Literal": {"Value": "'" + str(v).replace("'", "''") + "'"}}}
def col(t, p, name=None):
    d = {"field": {"Column": {"Expression": {"SourceRef": {"Entity": t}}, "Property": p}}, "queryRef": f"{t}.{p}", "nativeQueryRef": p}
    if name: d["displayName"] = name
    return d
AGG = {0: "Sum", 5: "Count", 1: "Avg"}
def agg(t, p, name=None, fn=0):
    d = {"field": {"Aggregation": {"Expression": {"Column": {"Expression": {"SourceRef": {"Entity": t}}, "Property": p}}, "Function": fn}},
         "queryRef": f"{AGG[fn]}({t}.{p})", "nativeQueryRef": f"{AGG[fn]} of {p}"}
    if name: d["displayName"] = name
    return d
counter = [0]
def add(page, vtype, pos, roles, title=None, objects=None, sort=None, extra_vco=None):
    counter[0] += 1
    name = f"{GEN}{counter[0]:02d}{uuid.uuid4().hex[:14]}"
    x, y, w, h = pos
    vis = {"visualType": vtype, "query": {"queryState": {r: {"projections": pr} for r, pr in roles.items()}}, "drillFilterOtherVisuals": True}
    if sort: vis["query"]["sortDefinition"] = {"sort": sort, "isDefaultSort": True}
    if objects: vis["objects"] = objects
    vco = {}
    if title: vco["title"] = [{"properties": {"show": lit(True), "text": lit(title)}}]
    if extra_vco: vco.update(extra_vco)
    if vco: vis["visualContainerObjects"] = vco
    body = {"$schema": SCHEMA, "name": name, "position": {"x": x, "y": y, "z": 5000 + counter[0], "height": h, "width": w, "tabOrder": 5000 + counter[0]}, "visual": vis}
    parts[f"definition/pages/{page}/visuals/{name}/visual.json"] = json.dumps(body, ensure_ascii=False, indent=2)
def mapobj(lat=49.43, lon=1.08, zoom=10.5):
    return {"mapControls": [{"properties": {"zoom": lit(zoom), "centerLatitude": lit(lat), "centerLongitude": lit(lon), "defaultStyle": lit("grayscale_dark")}}]}
def slicer_obj(mode="Dropdown", single=False, default=None, table=None, column=None):
    o = {"data": [{"properties": {"mode": lit(mode)}}]}
    if single: o["selection"] = [{"properties": {"singleSelect": lit(True), "strictSingleSelect": lit(True)}}]
    if default is not None:
        o["general"] = [{"properties": {"filter": {"filter": {"Version": 2, "From": [{"Name": "r", "Entity": table, "Type": 0}],
            "Where": [{"Condition": {"In": {"Expressions": [{"Column": {"Expression": {"SourceRef": {"Source": "r"}}, "Property": column}}],
                                             "Values": [[{"Literal": {"Value": "'" + default.replace("'", "''") + "'"}}]]}}}]}}}}]
    return o

CA, AC, OR, RR, RM = "carte_acces", "acces_communes", "options_rouen", "recommandations_rouen", "resultats_modele"
# ---------- Page 1 : Qui a acces ?
add(P1, "slicer", (24, 150, 440, 90), {"Values": [col(CA, "commune_nom", "Commune")]}, "Commune", slicer_obj("Dropdown"))
add(P1, "azureMap", (24, 256, 1100, 764), {"Category": [col(CA, "classe_acces", "Classe d'accès")], "Y": [col(CA, "lat")], "X": [col(CA, "lon")], "Size": [agg(CA, "population", "Habitants")]},
    "Habitants par carreau de 1 km, selon l'accès à une option de mobilité", mapobj())
add(P1, "donutChart", (1144, 150, 752, 420), {"Category": [col(CA, "classe_acces", "Classe d'accès")], "Y": [agg(CA, "population", "Habitants")]}, "Part des habitants par classe d'accès")
add(P1, "tableEx", (1144, 586, 752, 434), {"Values": [col(AC, "commune_nom", "Commune"), agg(AC, "options_par_habitant", "Options par habitant"), agg(AC, "part_sans_option", "Part sans option")]},
    "Communes au plus fort potentiel (moins d'options par habitant)", sort=[{"field": agg(AC, "options_par_habitant")["field"], "direction": "Ascending"}])
# ---------- Page 2 : Qui est concerne ?
add(P2, "azureMap", (24, 150, 1180, 870), {"Category": [col(OR, "type", "Type")], "Y": [col(OR, "lat")], "X": [col(OR, "lon")], "Size": [agg(OR, "pop_desservie", "Population desservie")]},
    "Options de mobilité existantes et population desservie", mapobj())
add(P2, "clusteredBarChart", (1224, 150, 672, 420), {"Category": [col(OR, "type", "Type")], "Y": [agg(OR, "pop_desservie", "Population desservie")]}, "Population desservie par type d'option")
add(P2, "tableEx", (1224, 586, 672, 434), {"Values": [col(OR, "nom", "Option"), col(OR, "type", "Type"), col(OR, "commune_nom", "Commune"), agg(OR, "pop_desservie", "Population desservie")]},
    "Options qui desservent le plus d'habitants", sort=[{"field": agg(OR, "pop_desservie")["field"], "direction": "Descending"}])
# ---------- Page 3 : Ou investir ?
add(P3, "slicer", (24, 150, 620, 100), {"Values": [col(RR, "scenario", "Scénario")]}, "Scénario",
    slicer_obj("Dropdown", True, "Base, coûts moyens", RR, "scenario"))
add(P3, "slicer", (660, 150, 420, 100), {"Values": [col(RR, "type", "Type")]}, "Type d'emplacement",
    slicer_obj("Dropdown", True, "Station Citiz", RR, "type"))
add(P3, "slicer", (1096, 150, 800, 100), {"Values": [col(RR, "cout_cumule", "Budget (coût cumulé, €)")]}, "Budget : coût cumulé maximal (€)", slicer_obj("Between"))
add(P3, "card", (24, 266, 340, 150), {"Values": [agg(RR, "habitants_gagnes", "Habitants gagnés")]}, "Habitants gagnés")
add(P3, "card", (380, 266, 340, 150), {"Values": [agg(RR, "rang", "Emplacements", 5)]}, "Emplacements financés")
add(P3, "card", (736, 266, 344, 150), {"Values": [agg(RR, "cout", "Coût total (€)")]}, "Coût total (€)")
add(P3, "azureMap", (24, 432, 1056, 588), {"Category": [col(RR, "type", "Type")], "Y": [col(RR, "lat")], "X": [col(RR, "lon")], "Size": [agg(RR, "habitants_gagnes", "Habitants gagnés")]},
    "Emplacements recommandés dans le budget", mapobj())
add(P3, "tableEx", (1096, 266, 800, 754), {"Values": [col(RR, "rang", "Rang"), col(RR, "type", "Type"), col(RR, "commune_nom", "Commune"), agg(RR, "habitants_gagnes", "Habitants gagnés"), agg(RR, "cout", "Coût (€)"), agg(RR, "cout_cumule", "Coût cumulé (€)")]},
    "Liste ordonnée des emplacements", sort=[{"field": col(RR, "rang")["field"], "direction": "Ascending"}])
# ---------- Page 4 : Le modele est-il fiable ?
add(P4, "clusteredBarChart", (24, 160, 980, 440), {"Category": [col(RM, "methode", "Méthode")], "Y": [agg(RM, "capture_top10", "Part des stations dans le top 10 %")]},
    "Part des vraies stations de Strasbourg retrouvées dans le top 10 % des carreaux")
add(P4, "tableEx", (24, 616, 1872, 404), {"Values": [col(RM, "methode", "Méthode"), agg(RM, "auc", "AUC"), agg(RM, "capture_top10", "Capture top 10 %"), agg(RM, "capture_top5", "Capture top 5 %"),
    agg(RM, "gain_vs_hasard_top10", "Gain vs hasard (top 10 %)"), agg(RM, "capture_top10_voisins", "Capture top 10 % (avec voisins)")]}, "Résultats du test sur Strasbourg")

pj = json.loads(parts["definition/pages/pages.json"]); pj["activePageName"] = P1
parts["definition/pages/pages.json"] = json.dumps(pj, indent=2)
body = {"definition": {"parts": [{"path": k, "payload": base64.b64encode(v.encode("utf-8")).decode(), "payloadType": "InlineBase64"} for k, v in parts.items() if k != ".platform"]}}
try:
    r = c.post(f"v1/workspaces/{ws}/reports/{rid}/updateDefinition", json=body, lro_wait=True)
    print("UPDATE", r.status_code, r.text[:500])
except Exception:
    print(traceback.format_exc()[-3000:])
print("generated visuals:", counter[0])